# Pandas - Manipulación de Datos

Este notebook cubre la manipulación de datos en Pandas: **selección y edición** de columnas y filas, **filtrado**, **sustitución de datos** (incl. datos vacíos/NaN), agregar/eliminar columnas, merge y join, groupby, y transformaciones con pivot y reshape.

## Selección de columnas y filas


In [5]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'nombre': ['Ana', 'Luis', 'Carla', 'Pedro', 'María'],
    'edad': [25, 30, 22, 35, 28],
    'ciudad': ['Madrid', 'Barcelona', None, 'Valencia', 'Madrid'],
    'ventas': [100, 150, 80, 200, 120]
}, index=[10, 20, 30, 40, 50])

# --- Selección de columnas ---
df['nombre']              # Una columna → Series
df[['nombre', 'edad']]    # Varias columnas → DataFrame
df.columns                # Lista de nombres de columnas

# --- Selección de filas por posición: iloc (índice entero) ---
df.iloc[0]                # Primera fila (Series)
df.iloc[0:2]              # Filas 0 y 1
df.iloc[:, 0]             # Primera columna (todas las filas)
df.iloc[1:4, [0, 2]]      # Filas 1-3, columnas 0 y 2
df.iloc[[0, 2, 4]]        # Filas 0, 2 y 4 (lista de posiciones)

# --- Selección por etiqueta: loc (nombres del índice/columnas) ---
df.loc[10]                # Fila con índice 10
df.loc[10:40]             # Filas desde índice 10 hasta 40 (inclusive)
df.loc[:, 'nombre']       # Columna 'nombre' (todas las filas)
df.loc[10, 'edad']       # Valor en fila 10, columna 'edad'
df.loc[[10, 30], ['nombre', 'ventas']]  # Subconjunto filas y columnas

# --- Selección de RANGOS de columnas y filas ---
# Rangos por posición (iloc): [inicio:fin] fin no incluido
df.iloc[1:4]              # Filas 1, 2 y 3 (rango de filas)
df.iloc[:, 0:3]           # Todas las filas, columnas 0, 1 y 2
df.iloc[0:2, 1:4]         # Filas 0-1, columnas 1, 2 y 3
df.iloc[1:-1]             # Filas del medio (excluye primera y última)
df.iloc[::2]              # Una de cada dos filas (paso 2)
df.iloc[:, ::2]           # Una de cada dos columnas

# Rangos por etiqueta (loc): [inicio:fin] fin SÍ incluido
df.loc[20:40]             # Filas con índice 20, 30 y 40 (inclusive)
df.loc[:, 'edad':'ventas']  # Columnas desde 'edad' hasta 'ventas' (inclusive)
df.loc[20:40, 'nombre':'ciudad']  # Rango de filas y rango de columnas

# Rango de columnas por posición usando .columns
df[df.columns[0:2]]       # Primeras 2 columnas
df[df.columns[1:4]]       # Columnas en posiciones 1, 2 y 3
df[df.columns[-2:]]       # Últimas 2 columnas

# --- Atajo: primera fila / últimas filas ---
df.head(2)                # Primeras 2 filas
df.tail(2)                # Últimas 2 filas
df.sample(2)              # 2 filas aleatorias

,nombre,edad,ciudad,ventas
10,Ana,25,Madrid,100
30,Carla,22,None,80


## Edición de columnas y filas

Renombrar columnas o índices, modificar valores concretos y crear columnas a partir de otras.

In [6]:
# Renombrar columnas
df.rename(columns={'ventas': 'total_ventas', 'edad': 'años'})
df.rename(columns=str.upper)   # Todas a mayúsculas
df.rename(index={10: 'id_10', 20: 'id_20'})

# Modificar valores por posición o etiqueta
df.loc[10, 'edad'] = 26           # Una celda
df.loc[10:30, 'ventas'] = 0       # Rango de celdas
df.iloc[0, 1] = 99                # Por posición

# Crear o sobrescribir columnas
df['bonus'] = df['ventas'] * 0.1           # Columna calculada
df['region'] = 'Norte'                      # Mismo valor para todas las filas
df.assign(ventas_x2=df['ventas'] * 2)      # Nuevo DataFrame con columna extra (no modifica df)

# Reemplazar valores concretos (sustitución por valor)
df.replace('Madrid', 'MAD')                 # Todas las celdas
df.replace({'ciudad': {'Madrid': 'MAD', 'Barcelona': 'BCN'}})  # Por columna
df['edad'].replace(25, 26)                  # En una columna
df.replace([100, 150], [101, 151])          # Lista de valores → lista de reemplazos

,nombre,edad,ciudad,ventas,bonus,region
10,Ana,99,Madrid,0,0.0,Norte
20,Luis,30,Barcelona,0,0.0,Norte
30,Carla,22,None,0,0.0,Norte
40,Pedro,35,Valencia,200,20.0,Norte
50,María,28,Madrid,120,12.0,Norte


### Modificar sobrescribiendo (in-place) y copias

Dos formas de que los cambios “queden”: **reasignar** la variable o usar **inplace**. Para trabajar sin alterar el original, usar **copia**.

In [7]:
# --- Sobrescribir el DataFrame (que los cambios persistan) ---
# DataFrame de ejemplo con columna 'C' para los ejemplos de drop
df_sob = pd.DataFrame({'A': [1, 2], 'B': [4, 5], 'C': [7, 8], 'ventas': [100, 150]})

# Opción 1: Reasignar la variable (la mayoría de métodos devuelven un nuevo DataFrame)
df_sob = df_sob.drop('C', axis=1)           # df_sob pasa a ser el resultado de quitar la columna
df_sob = df_sob.rename(columns={'ventas': 'total_ventas'})
df_sob = df_sob.fillna(0)

# Opción 2: inplace=True (modifica el mismo objeto, no devuelve nada)
df_in = pd.DataFrame({'A': [1, 2], 'B': [4, 5], 'C': [7, 8], 'ventas': [100, 150]})
df_in.drop('C', axis=1, inplace=True)
df_in.rename(columns={'ventas': 'total_ventas'}, inplace=True)
df_in.fillna(0, inplace=True)

# Asignación directa con loc/iloc SÍ modifica el DataFrame original (usamos df de la celda anterior)
df.loc[10, 'edad'] = 26             # Cambia el objeto df
df['nueva_col'] = 1                 # Añade columna al mismo df

# --- Copia: trabajar sin modificar el original ---
df_original = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})

# Copia superficial: .copy()
df_copia = df_original.copy()       # Nuevo DataFrame con los mismos datos
df_copia['A'] = 0                   # Solo cambia df_copia; df_original no se toca

# Sin .copy(), una “copia” por asignación es solo otra referencia al mismo objeto
df_falso = df_original              # Mismo objeto
df_falso['A'] = 0                   # ¡También cambia df_original!

# Copia profunda (si hay objetos anidados que no quieres compartir)
df_copia_profunda = df_original.copy(deep=True)

## Filtrado de datos

Obtener subconjuntos de filas según condiciones (máscaras booleanas) o con `query()`.

In [8]:
# Máscara booleana (Series de True/False)
df['edad'] > 25
df['ciudad'] == 'Madrid'
df['ventas'].between(90, 130)

# Filtrar filas con máscara
df[df['edad'] > 25]                      # Solo filas con edad > 25
df[df['ciudad'] == 'Madrid']             # Ciudad exacta
df[df['ventas'] >= 100]                  # Ventas >= 100

# Condiciones combinadas: & (y), | (o), ~ (no)
df[(df['edad'] > 25) & (df['ventas'] > 100)]
df[(df['ciudad'] == 'Madrid') | (df['ciudad'] == 'Barcelona')]
df[~df['ciudad'].isna()]                 # Filas donde ciudad no es NaN

# isin: filtrar por lista de valores
df[df['ciudad'].isin(['Madrid', 'Valencia'])]
df[df['edad'].isin([25, 30, 35])]

# query(): expresiones como strings (útil para columnas con espacios)
df.query('edad > 25 and ventas >= 100')
df.query('ciudad in ["Madrid", "Barcelona"]')
df.query('edad >= 22 and edad <= 30')

# Filtrar por valores no nulos / nulos
df[df['ciudad'].notna()]
df[df['ciudad'].isna()]                  # Filas con ciudad faltante

,nombre,edad,ciudad,ventas,bonus,region,nueva_col
30,Carla,22,None,0,0.0,Norte,1


## Sustitución y datos faltantes (NaN)

Rellenar o eliminar valores vacíos (`NaN`, `None`) y sustituir valores por otros (p. ej. constantes, media, relleno hacia adelante/atrás).

In [9]:
# DataFrame de ejemplo con faltantes
df_na = pd.DataFrame({
    'A': [1, np.nan, 3, np.nan, 5],
    'B': [10, 20, np.nan, 40, 50],
    'C': ['x', None, 'z', None, 'x']
})

# Detectar faltantes
df_na.isna()           # True donde hay NaN/None
df_na.notna()          # True donde no hay faltantes
df_na.isna().sum()     # Conteo de NaN por columna

# Rellenar valores faltantes: fillna()
df_na.fillna(0)                    # Sustituir todos los NaN por 0
df_na['A'].fillna(df_na['A'].mean())  # Columna A: NaN → media de A
df_na.fillna({'A': 0, 'B': 999, 'C': 'N/A'})  # Valor distinto por columna
df_na.ffill()                      # Forward fill: rellenar con el valor anterior
df_na.bfill()                      # Backward fill: con el siguiente
df_na.ffill(limit=1)               # Solo 1 relleno consecutivo

# Eliminar filas/columnas con faltantes: dropna()
df_na.dropna()                     # Eliminar filas con algún NaN
df_na.dropna(axis=1)               # Eliminar columnas con algún NaN
df_na.dropna(how='all')            # Solo filas donde toda la fila es NaN
df_na.dropna(subset=['A', 'B'])   # Eliminar filas con NaN en A o B
df_na.dropna(thresh=2)             # Mantener filas con al menos 2 valores no nulos

# Sustituir valores arbitrarios (no solo NaN): replace()
df_na.replace(np.nan, 0)          # NaN → 0
df_na.replace({np.nan: 0, None: 'vacío'})
df_na['C'].replace('x', 'X')      # En una columna
df_na.replace([1, 10], [100, 200])  # 1→100, 10→200

,A,B,C
0,100.0,200.0,x
1,NaN,20.0,None
2,3.0,NaN,z
3,NaN,40.0,None
4,5.0,50.0,x


## Manipulación de Índice


In [10]:
import pandas as pd

# Ejemplo de DataFrame para manipulación de índice
df_idx = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]}, index=['a', 'b', 'c'])

# Reindexar (conformar a nuevo índice)
df_idx.reindex(['a', 'b', 'c', 'd'])  # Nuevo índice (rellena con NaN si falta)

# Eliminar labels del índice
df_idx.drop(labels=['a'])             # Eliminar labels del índice

# Renombrar labels del índice
df_idx.rename(index={'a': 'A', 'b': 'B'})  # Renombrar labels del índice

# Ordenar índice
df_idx.sort_index()                   # Ordenar labels del índice

# Establecer columna como índice
df_idx.set_index('A')                 # Insertar columna en índice
# Para MultiIndex, necesitaríamos un DataFrame con más columnas:
df_multi = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6], 'C': [7, 8, 9]}, index=['a', 'b', 'c'])
df_multi.set_index(['A', 'B'])        # MultiIndex desde columnas (A y B pasan a ser índice)

# Resetear índice
df_idx.reset_index()                  # Insertar índice en columnas, resetear a RangeIndex


,index,A,B
0,a,1,4
1,b,2,5
2,c,3,6


## Agregar y Eliminar Columnas

### Eliminar filas y columnas

Usar `drop()` para quitar columnas o filas por nombre/etiqueta. Por defecto devuelve un nuevo DataFrame; con `inplace=True` se modifica el original.

In [11]:
import pandas as pd

df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6], 'C': [7, 8, 9]}, index=['a', 'b', 'c'])

# --- Eliminar COLUMNAS ---
df.drop('C', axis=1)                  # Una columna → axis=1 o axis='columns'
df.drop(columns='C')                 # Equivalente (más legible)
df.drop(['B', 'C'], axis=1)           # Varias columnas
df.drop(columns=['B', 'C'])          # Varias columnas (recomendado)

# --- Eliminar FILAS ---
df.drop('a')                         # Una fila (por etiqueta del índice)
df.drop(index='a')                   # Equivalente
df.drop(['a', 'c'])                  # Varias filas por etiqueta
df.drop(index=['a', 'c'])
df.drop(labels=['a', 'b'], axis=0)   # axis=0 o axis='rows'

# Eliminar por posición (usar índices enteros)
df.drop(df.index[0])                 # Primera fila
df.drop(df.index[1:3])               # Filas en posiciones 1 y 2
df.drop(df.index[[0, 2]])           # Filas 0 y 2

# --- Errores si la etiqueta no existe ---
# Por defecto da error si intentas eliminar una columna/fila que no existe.
df.drop('X', axis=1, errors='ignore')  # Si 'X' no existe, no hace nada (evita error)
df.drop('x', errors='ignore')          # Igual para filas

# --- Aplicar los cambios (sobrescribir o inplace) ---
df = df.drop('C', axis=1)            # Reasignar: df queda sin la columna
df2 = pd.DataFrame({'A': [1, 2], 'B': [4, 5], 'C': [7, 8]})
df2.drop('C', axis=1, inplace=True)  # Modificar el mismo DataFrame (inplace)

In [12]:
import pandas as pd

# Crear DataFrame inicial
df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]}, index=['a', 'b', 'c'])

# Añadir columna
df['C'] = [7, 8, 9]
df['D'] = df['A'] + df['B']  # Columna calculada
df['new_or_old_col'] = pd.Series([10, 11, 12], index=df.index)  # Desde Series o array

# Eliminar columnas
df_sin_c = df.drop('C', axis=1)              # Retorna nuevo DataFrame
df_sin_c = df.drop('C', axis='columns')      # Equivalente
df.rename(columns={'C': 'C_new'})           # Renombrar columna
df.drop(['C', 'D'], axis=1)                  # Eliminar múltiples columnas

# Para modificar in-place, crear nuevo DataFrame
df2 = df.copy()
df2.drop('C', axis=1, inplace=True)  # Modifica in-place

# Eliminar filas
df_sin_fila0 = df.drop('a')                  # Eliminar fila por índice
df_sin_filas = df.drop(['a', 'b'], axis=0)   # Múltiples filas
df.drop(labels=['a', 'b'])                   # Especificar labels explícitamente


,A,B,C,D,new_or_old_col
c,3,6,9,9,12


## Merge y Join


In [13]:
df1 = pd.DataFrame({'key': ['A', 'B', 'C'], 'value1': [1, 2, 3]})
df2 = pd.DataFrame({'key': ['B', 'C', 'D'], 'value2': [4, 5, 6]})

# Merge (similar a SQL JOIN)
pd.merge(df1, df2, on='key')                    # Inner join
pd.merge(df1, df2, on='key', how='left')        # Left join
pd.merge(df1, df2, on='key', how='right')      # Right join
pd.merge(df1, df2, on='key', how='outer')      # Outer join

# Join por índice (requiere que los índices coincidan o usar set_index)
df1_indexed = df1.set_index('key')
df2_indexed = df2.set_index('key')
df1_indexed.join(df2_indexed, how='inner')

# Concatenar
pd.concat([df1, df2], axis=0)    # Apilar verticalmente
pd.concat([df1, df2], axis=1)    # Apilar horizontalmente


,key,value1,key,value2
0,A,1,B,4
1,B,2,C,5
2,C,3,D,6


## GroupBy


In [14]:
df = pd.DataFrame({
    'categoria': ['A', 'A', 'B', 'B', 'A'],
    'valor': [10, 20, 30, 40, 50]
})

# Agrupación básica
grupo = df.groupby('categoria')
grupo.sum()           # Suma por grupo
grupo.mean()          # Media por grupo
grupo.count()         # Conteo por grupo

# Múltiples funciones de agregación
grupo.agg(['sum', 'mean', 'count'])

# Agrupar por múltiples columnas (ejemplo con DataFrame que tiene múltiples columnas)
df2 = pd.DataFrame({
    'categoria': ['A', 'A', 'B', 'B', 'A'],
    'subcategoria': ['X', 'Y', 'X', 'Y', 'X'],
    'valor': [10, 20, 30, 40, 50]
})
df2.groupby(['categoria', 'subcategoria']).sum()

# Aplicar función personalizada
grupo.apply(lambda x: x.max() - x.min())


,valor
categoria,
A,40
B,10


## Diferencia entre sum() y count()

**`sum()`**: Suma los valores numéricos de cada grupo. Ignora valores `NaN` por defecto.

**`count()`**: Cuenta el número de valores no nulos (no-NaN) en cada grupo.

**Diferencias clave:**
- `sum()` realiza una operación matemática (suma) sobre valores numéricos
- `count()` cuenta elementos, no los suma
- `sum()` ignora `NaN` en el cálculo (no los incluye en la suma)
- `count()` cuenta solo valores no nulos (excluye `NaN` del conteo)

In [15]:
import pandas as pd
import numpy as np

# Ejemplo para entender la diferencia entre sum() y count()
df_ejemplo = pd.DataFrame({
    'categoria': ['A', 'A', 'A', 'B', 'B', 'B'],
    'valor': [10, 20, np.nan, 30, 40, 50]
})

print("DataFrame original:")
print(df_ejemplo)
print("\n" + "="*50 + "\n")

# Agrupación
grupo = df_ejemplo.groupby('categoria')

# sum() - Suma los valores numéricos (ignora NaN)
print("sum() - Suma de valores numéricos:")
print(grupo.sum())
print("\nCategoría A: 10 + 20 + NaN = 30 (NaN ignorado)")
print("Categoría B: 30 + 40 + 50 = 120")
print("\n" + "="*50 + "\n")

# count() - Cuenta valores no nulos
print("count() - Conteo de valores no nulos:")
print(grupo.count())
print("\nCategoría A: 2 valores no nulos (10, 20; NaN no se cuenta)")
print("Categoría B: 3 valores no nulos (30, 40, 50)")
print("\n" + "="*50 + "\n")

# Comparación lado a lado
print("Comparación lado a lado:")
resultado = pd.DataFrame({
    'sum': grupo.sum()['valor'],
    'count': grupo.count()['valor']
})
print(resultado)

DataFrame original:
  categoria  valor
0         A   10.0
1         A   20.0
2         A    NaN
3         B   30.0
4         B   40.0
5         B   50.0


sum() - Suma de valores numéricos:
           valor
categoria       
A           30.0
B          120.0

Categoría A: 10 + 20 + NaN = 30 (NaN ignorado)
Categoría B: 30 + 40 + 50 = 120


count() - Conteo de valores no nulos:
           valor
categoria       
A              2
B              3

Categoría A: 2 valores no nulos (10, 20; NaN no se cuenta)
Categoría B: 3 valores no nulos (30, 40, 50)


Comparación lado a lado:
             sum  count
categoria              
A           30.0      2
B          120.0      3


## Ordenamiento de Valores


In [16]:
df = pd.DataFrame({
    'X': [3, 1, 2],
    'Y': [1, 3, 2]
}, index=['a', 'b', 'c'])

# Ordenar por valores de columna
df.sort_values('X', ascending=True)              # Ordenar por una columna
df.sort_values(['X', 'Y'], ascending=[False, True])  # Ordenar por múltiples columnas
# Todos los valores de fila e índice seguirán el ordenamiento


,X,Y
a,3,1
c,2,2
b,1,3


## Pivot y Reshape


In [17]:
df = pd.DataFrame({
    'fecha': ['2024-01', '2024-01', '2024-02', '2024-02'],
    'producto': ['A', 'B', 'A', 'B'],
    'ventas': [100, 200, 150, 250]
})

# Pivot
df.pivot(index='fecha', columns='producto', values='ventas')

# Pivot table (permite duplicados)
df.pivot_table(values='ventas', index='fecha', columns='producto', aggfunc='sum')

# Melt (inverso de pivot)
df.melt(id_vars='fecha', value_vars=['producto', 'ventas'])

# Stack y Unstack
df.set_index(['fecha', 'producto']).unstack()


ventas     
producto      A    B
fecha               
2024-01     100  200
2024-02     150  250